# Reconnaissance automatique des espèces menacées\n\nProjet AI for Good avec données imaginées.\n\nObjectif : entraîner un modèle capable de reconnaître une espèce menacée probable à partir de caractéristiques observables.

In [ ]:
!pip install pandas numpy scikit-learn matplotlib joblib -q

In [ ]:
import numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay\nimport joblib

## 1. Création de données fictives

In [ ]:
profiles = {\n    'Panda roux': ('Himalaya', 'forêt tempérée', 'crépusculaire', 'roux et blanc', (50,70), (3,7), (1800,4000), 'en danger'),\n    'Tigre de Sumatra': ('Sumatra', 'forêt tropicale', 'nocturne', 'rayures noires', (220,260), (75,140), (0,1200), 'critique'),\n    'Gorille des montagnes': ('Afrique centrale', 'forêt montagneuse', 'diurne', 'pelage sombre', (140,180), (70,180), (2200,4300), 'en danger'),\n    'Tortue imbriquée': ('océans tropicaux', 'récif corallien', 'diurne', 'carapace écailleuse', (60,100), (40,80), (0,5), 'critique'),\n    'Vaquita': ('Golfe de Californie', 'mer côtière', 'diurne', 'gris avec taches', (120,150), (30,55), (0,2), 'critique'),\n    'Rhinocéros de Java': ('Java', 'forêt humide', 'crépusculaire', 'peau grise', (300,340), (900,2300), (0,600), 'critique'),\n    'Léopard des neiges': ('Asie centrale', 'montagne rocheuse', 'crépusculaire', 'rosettes grises', (160,230), (25,55), (3000,5500), 'vulnérable'),\n    'Orang-outan de Bornéo': ('Bornéo', 'forêt tropicale', 'diurne', 'pelage orangé', (110,150), (30,100), (0,1500), 'critique'),\n}\nrng = np.random.default_rng(42)\nrows = []\nfor species, p in profiles.items():\n    region, habitat, activity, pattern, length, weight, altitude, threat = p\n    for _ in range(120):\n        rows.append({\n            'species': species, 'region': region, 'habitat': habitat, 'activity': activity,\n            'visual_pattern': pattern, 'threat_level': threat,\n            'body_length_cm': round(rng.uniform(*length) + rng.normal(0,4), 2),\n            'weight_kg': round(max(0.5, rng.uniform(*weight) + rng.normal(0,2)), 2),\n            'altitude_m': round(max(0, rng.uniform(*altitude) + rng.normal(0,80)), 2),\n            'temperature_c': round(rng.uniform(5,34), 2),\n            'observation_quality': rng.choice(['faible', 'moyenne', 'bonne'], p=[0.15,0.35,0.50])\n        })\ndf = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)\ndf.head()

## 2. Entraînement du modèle

In [ ]:
numeric_features = ['body_length_cm', 'weight_kg', 'altitude_m', 'temperature_c']\ncategorical_features = ['region', 'habitat', 'activity', 'visual_pattern', 'threat_level', 'observation_quality']\nX = df[numeric_features + categorical_features]\ny = df['species']\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)\npreprocessor = ColumnTransformer([\n    ('num', StandardScaler(), numeric_features),\n    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),\n])\nmodel = Pipeline([\n    ('preprocessor', preprocessor),\n    ('classifier', RandomForestClassifier(n_estimators=250, random_state=42, class_weight='balanced'))\n])\nmodel.fit(X_train, y_train)\npreds = model.predict(X_test)\nprint(classification_report(y_test, preds))

## 3. Visualisation

In [ ]:
cm = confusion_matrix(y_test, preds, labels=model.classes_)\ndisp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)\nfig, ax = plt.subplots(figsize=(12, 8))\ndisp.plot(ax=ax, xticks_rotation=45)\nplt.title('Matrice de confusion')\nplt.show()

## 4. Prédiction d'une observation

In [ ]:
observation = pd.DataFrame([{\n    'body_length_cm': 185,\n    'weight_kg': 42,\n    'altitude_m': 4200,\n    'temperature_c': 8,\n    'region': 'Asie centrale',\n    'habitat': 'montagne rocheuse',\n    'activity': 'crépusculaire',\n    'visual_pattern': 'rosettes grises',\n    'threat_level': 'vulnérable',\n    'observation_quality': 'bonne',\n}])\nprediction = model.predict(observation)[0]\nconfidence = model.predict_proba(observation).max()\nprint('Espèce prédite :', prediction)\nprint('Confiance :', f'{confidence:.1%}')

## Note éthique\nPour un vrai projet, il faut protéger les localisations exactes des espèces menacées afin d'éviter le braconnage.